# Module 01: agentic-rag

## 02-environment

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

## 03-rag

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s up with you?'

In [7]:
question = "I just discovered the course. Can I join now?"

In [6]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—possibly, but it depends on the course’s enrollment policy and whether there’s still room.

If you mean a specific course, check:
- whether late enrollment is allowed
- the registration deadline
- whether there are prerequisites
- if the class has a waitlist

If you want, send me the course name or a link and I can help you figure out whether you can still join and what to do next.


In [5]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still being accepted.


## 04-dataset

In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [11]:
for course in courses_raw[0]:
    print(course)

course
course_name
path
questions_count


(my notes:) nice way of using extended() method to avoid using append() and sum(list_obj, [])

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for n, course in enumerate(courses_raw):
        
    course_url = f"""{url_prefix}{course["path"]}"""
    print(course_url)

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)


len(documents)

https://datatalks.club/faq/json/data-engineering-zoomcamp.json
https://datatalks.club/faq/json/stock-markets-analytics-zoomcamp.json
https://datatalks.club/faq/json/ai-dev-tools-zoomcamp.json
https://datatalks.club/faq/json/llm-zoomcamp.json
https://datatalks.club/faq/json/mlops-zoomcamp.json
https://datatalks.club/faq/json/machine-learning-zoomcamp.json


1350

In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

## 05-search

In [14]:
from minsearch import Index

In [15]:
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [33]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

We see questions about joining the course, registration, certificates, and more. These are the candidate documents we'll send to the LLM.

We used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). Query words appearing in the question field is a stronger signal than them appearing in the section name.

We used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.

**Boosting fields**

Not all fields are equally important. The question field is usually more relevant than section for matching. Query words appearing in the question is a stronger signal than them appearing in the section name.

minsearch supports field boosting to reflect this:

In [17]:
results = index.search(
    question,
    num_results=2,
    boost_dict={"question": 2.0, "section": 0.5}
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}]

All fields have a default boost of 1. Giving question a boost of 2 means it counts two times as much. Take a question about certificates. The word "certificate" in the question field now weighs twice what it does in the answer.

Giving section 0.5 means it counts half as much, since a match there tells us less. This is the same boosting mechanism used by Elasticsearch and Lucene.

**Filtering by course**

Sometimes you want to restrict the search to a specific course.

minsearch supports keyword filtering

In [18]:
results = index.search(
    question,
    num_results=3,
    # filter_dict={"course": "mlops-zoomcamp"}
    filter_dict={"course": "stock-markets-analytics-zoomcamp"}
)

results

[{'id': 'bbf72858d3',
  'course': 'stock-markets-analytics-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I start now or should I wait for the official launch?',
  'answer': "You can start now by going through last year's GitHub repo (recordings, slides, homeworks, code). About 80% of the materials stay the same, so reviewing them early shows you what to expect and gives you time to brush up on Python/Pandas before the launch."},
 {'id': '265f497dea',
  'course': 'stock-markets-analytics-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Where can I find the course repo with the slides and code?',
  'answer': 'The repo link is shared in nearly every announcement via Telegram, Slack and email - use that link for the slides and code.'},
 {'id': '5ea7790727',
  'course': 'stock-markets-analytics-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can you show an example Streamlit dashboard from the course?',
  

**Wrapping it in a function**

In [19]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [20]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## 06-building-prompt

In [21]:
#The instructions tell the LLM its role and how to answer:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

#This is what grounds the answer in our data and reduces hallucinations.

In [22]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [23]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

#Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM.

In [24]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [25]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

## 07-llm

In [26]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [27]:
response.output[0]

ResponseOutputMessage(id='msg_0dea95ebc90bf667006a398ce26be481a2807fafe5a3e9bd53', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, and the course must be running live for you to earn it.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [28]:
response.output[0].content[0].text

'Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, and the course must be running live for you to earn it.'

In [29]:
response.output_text

'Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, and the course must be running live for you to earn it.'

In [30]:
response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=43, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=523)

In [31]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0005535000000000001

We send two messages:

- developer - system-level instructions (how the LLM should behave)
- user - the actual prompt with the question and context


In [35]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

# This separates the fixed instructions from the user prompt, which changes every request.

In [38]:
response.output_text

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

OpenAI accepts both `developer` and `system` for the instruction role. There may be some difference between them, but in practice I don't see it change the result either way. We use developer in this course.

In [36]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [39]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [40]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still being accepted.


In [41]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a **live cohort**.\n\nYou **cannot** get a certificate in **self-paced** mode, because the certificate requires you to **peer-review 3 capstone projects** after submitting your own project, and that peer-review step is only available while the course is running.\n\nTo make sure your certificate shows the right name, set your **official name** in your course profile.'